# Install Dependecies

In [1]:

%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [37]:
import pandas as pd
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import classification_report
import ipynbname
from pathlib import Path

In [38]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/nlp/msc-nlp-2026/project_notebooks


# Import datasets

In [4]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [5]:
# tokeniser
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

### Pick languages

In [6]:
LANGUAGES = ["ar", "ko", "te"]

In [7]:
# select languages
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# 2 Week 36: Data and Rule-Based Baselines
Download the dataset and inspect its columns. Report Item 1 separately by
language and split, plus overall example counts and answerability proportions.
Compute Item 2 from the training questions separately by language. Apply
Item 3 to every answerable example in both splits.

Implement and evaluate two answerability baselines: (i) the majority-class
baseline estimated from the training split and (ii) a deterministic rule-based
classifier that uses only the question and context. The rule may use tokenisation,
lexical features or machine translation, but no labelled validation examples or
trained answerability/QA model. Discuss what information the rule can and
cannot exploit in this cross-lingual setting.

## Week 36.1
Report the number of examples, answerable/unanswerable proportions, median and interquartile range of tokenised question and context lengths (a table is sufficient; plots are optional), missing values and exact duplicate question–context pairs.

In [8]:
def token_len(texts):
    return [len(xlm_tokeniser(text)["input_ids"]) for text in texts]

In [9]:
df_train["q_len"] = token_len(df_train["question"])
df_train["c_len"] = token_len(df_train["context"])
df_val["q_len"] = token_len(df_val["question"])
df_val["c_len"] = token_len(df_val["context"])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (549 > 512). Running this sequence through the model will result in indexing errors


In [10]:
df_train.head(5)

,question,context,lang,answerable,answer_start,answer,answer_inlang,q_len,c_len
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,11,163
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None,12,246
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None,17,273
4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None,11,107
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None,12,123


### Compute the number of answerable / unanswerable / pct_unanswerable

In [11]:
# train
counts = df_train.groupby(["lang"])["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(0)
print(counts)

         n  answerable  unanswerable  pct_unanswerable
lang                                                  
ar    2558        2303           255              10.0
ko    2422        2359            63               3.0
te    1355        1310            45               3.0


In [12]:
# val
counts = df_val.groupby("lang")["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(0)
print(counts)

        n  answerable  unanswerable  pct_unanswerable
lang                                                 
ar    415         363            52              13.0
ko    356         337            19               5.0
te    384         291            93              24.0


### Overall counts

In [13]:
# train
n = len(df_train)
answerable = df_train["answerable"].sum()
print("n:", n, "| answerable:", answerable, "| unanswerable:", n - answerable,
      "| pct_unanswerable:", round(100 * (n - answerable) / n, 2))

n: 6335 | answerable: 5972 | unanswerable: 363 | pct_unanswerable: 5.73


In [14]:
# val
n = len(df_val)
answerable = df_val["answerable"].sum()
print("n:", n, "| answerable:", answerable, "| unanswerable:", n - answerable,
      "| pct_unanswerable:", round(100 * (n - answerable) / n, 2))

n: 1155 | answerable: 991 | unanswerable: 164 | pct_unanswerable: 14.2


### Median and IQR of question / context length

In [15]:
# train
grouped = df_train.groupby("lang")
lengths = pd.DataFrame({
    "q_q25": grouped["q_len"].quantile(0.25),
    "q_median": grouped["q_len"].median(),
    "q_q75": grouped["q_len"].quantile(0.75),
    "c_q25": grouped["c_len"].quantile(0.25),
    "c_median": grouped["c_len"].median(),
    "c_q75": grouped["c_len"].quantile(0.75),
})
lengths["q_iqr"] = lengths["q_q75"] - lengths["q_q25"]
lengths["c_iqr"] = lengths["c_q75"] - lengths["c_q25"]
print(lengths)

      q_q25  q_median  q_q75  c_q25  c_median  c_q75  q_iqr  c_iqr
lang                                                              
ar     11.0      13.0   15.0   89.0     133.0  191.0    4.0  102.0
ko     12.0      14.0   16.0   85.0     127.0  181.0    4.0   96.0
te     11.0      13.0   16.0   80.0     121.0  171.0    5.0   91.0


In [16]:
# val
grouped = df_val.groupby("lang")
lengths = pd.DataFrame({
    "q_q25": grouped["q_len"].quantile(0.25),
    "q_median": grouped["q_len"].median(),
    "q_q75": grouped["q_len"].quantile(0.75),
    "c_q25": grouped["c_len"].quantile(0.25),
    "c_median": grouped["c_len"].median(),
    "c_q75": grouped["c_len"].quantile(0.75),
})
lengths["q_iqr"] = lengths["q_q75"] - lengths["q_q25"]
lengths["c_iqr"] = lengths["c_q75"] - lengths["c_q25"]
print(lengths)

      q_q25  q_median  q_q75   c_q25  c_median  c_q75  q_iqr   c_iqr
lang                                                                
ar     11.0      12.0   15.0   89.50     131.0  190.5    4.0  101.00
ko     12.0      14.0   16.0   80.75     123.5  183.5    4.0  102.75
te     12.0      14.0   18.0  114.00     154.0  208.0    6.0   94.00


In [17]:
def quantile_metrics(s):
    return [s.quantile(0.0).item(), s.quantile(0.25).item(), s.median().item(), s.quantile(0.75).item(), s.quantile(1.0).item()]

In [18]:
def print_quantile_metrics(col):
    for lang in df_train["lang"].unique():
        print(lang)
        print(f"\tTraining dataset quantiles:\t {quantile_metrics(df_train[df_train['lang'] == lang][col])}")
        print(f"\tValidation dataset quantiles:\t {quantile_metrics(df_val[df_val['lang'] == lang][col])}")

In [19]:
# question length
print_quantile_metrics("q_len")

ko
	Training dataset quantiles:	 [6.0, 12.0, 14.0, 16.0, 47.0]
	Validation dataset quantiles:	 [7.0, 12.0, 14.0, 16.0, 34.0]
ar
	Training dataset quantiles:	 [6.0, 11.0, 13.0, 15.0, 31.0]
	Validation dataset quantiles:	 [6.0, 11.0, 12.0, 15.0, 65.0]
te
	Training dataset quantiles:	 [7.0, 11.0, 13.0, 16.0, 33.0]
	Validation dataset quantiles:	 [8.0, 12.0, 14.0, 18.0, 26.0]


In [20]:
# context length
print_quantile_metrics("c_len")

ko
	Training dataset quantiles:	 [12.0, 85.0, 127.0, 181.0, 1528.0]
	Validation dataset quantiles:	 [20.0, 80.75, 123.5, 183.5, 623.0]
ar
	Training dataset quantiles:	 [13.0, 89.0, 133.0, 191.0, 1154.0]
	Validation dataset quantiles:	 [24.0, 89.5, 131.0, 190.5, 861.0]
te
	Training dataset quantiles:	 [11.0, 80.0, 121.0, 171.0, 819.0]
	Validation dataset quantiles:	 [12.0, 114.0, 154.0, 208.0, 589.0]


### Missing values

In [21]:
print(df_train.isna().sum().to_markdown())
print("-------------------------")
print(df_val.isna().sum().to_markdown())

|               |    0 |
|:--------------|-----:|
| question      |    0 |
| context       |    0 |
| lang          |    0 |
| answerable    |    0 |
| answer_start  |    0 |
| answer        |    0 |
| answer_inlang | 6285 |
| q_len         |    0 |
| c_len         |    0 |
-------------------------
|               |    0 |
|:--------------|-----:|
| question      |    0 |
| context       |    0 |
| lang          |    0 |
| answerable    |    0 |
| answer_start  |    0 |
| answer        |    0 |
| answer_inlang | 1055 |
| q_len         |    0 |
| c_len         |    0 |


### Duplicate pairs

In [22]:
print("train:", df_train.duplicated(subset=["question", "context"]).sum())
print("val:", df_val.duplicated(subset=["question", "context"]).sum())

train: 10
val: 0


## Week 36.2
Report the five most common question tokens and their counts for each language, together with an English translation, and explain your tokenisation.

In [23]:
def n_common_tokens(questions, n=5):
    tokens = []
    for q in questions:
        tokens += xlm_tokeniser.tokenize(q)
    return Counter(tokens).most_common(n)

In [24]:
# train
rows = []
for lang in df_train["lang"].unique():
    questions = df_train.loc[df_train["lang"] == lang, "question"]
    for token, count in n_common_tokens(questions):
        rows.append({"lang": lang, "token": token, "count": count})

In [25]:
translations = {
    0: "question mark",
    1: "topic particle (after vowel)",
    2: "word boundary",
    3: "topic particle (after consonant)",
    4: "subject particle (after vowel)",
    5: "Arabic question mark",
    6: "question mark after space",
    7: "word boundary",
    8: "word-initial m- (ما/من/متى: what/who/when)",
    9: "in",
    10: "question mark",
    11: "word boundary",
    12: "who",
    13: "question mark after space",
    14: "in (locative suffix)",
}

df_tokens = pd.DataFrame(rows)
df_tokens["translation"] = pd.Series(translations)
display(df_tokens)

,lang,token,count,translation
0,ko,?,2420,question mark
1,ko,는,1154,topic particle (after vowel)
2,ko,▁,986,word boundary
3,ko,은,985,topic particle (after consonant)
4,ko,가,691,subject particle (after vowel)
5,ar,؟,1499,Arabic question mark
6,ar,▁؟,1057,question mark after space
7,ar,▁,653,word boundary
8,ar,▁م,624,word-initial m- (ما/من/متى: what/who/when)
9,ar,▁في,619,in


In [26]:
# val
rows = []
for lang in df_val["lang"].unique():
    questions = df_val.loc[df_val["lang"] == lang, "question"]
    for token, count in n_common_tokens(questions):
        rows.append({"lang": lang, "token": token, "count": count})

In [27]:
translations_val = {
    0: "question mark",
    1: "word boundary",
    2: "in (locative suffix)",
    3: "which / what",
    4: "country",
    5: "question mark",
    6: "topic particle (after vowel)",
    7: "topic particle (after consonant)",
    8: "word boundary",
    9: "subject particle (after vowel)",
    10: "Arabic question mark",
    11: "question mark after space",
    12: "who",
    13: "word boundary",
    14: "in",
}

df_tokens_val = pd.DataFrame(rows)
assert df_tokens_val["lang"].tolist() == ["te"]*5 + ["ko"]*5 + ["ar"]*5
df_tokens_val["translation"] = pd.Series(translations_val)
display(df_tokens_val)

,lang,token,count,translation
0,te,?,315,question mark
1,te,▁,146,word boundary
2,te,లో,109,in (locative suffix)
3,te,▁ఏ,96,which / what
4,te,దేశం,85,country
5,ko,?,356,question mark
6,ko,는,157,topic particle (after vowel)
7,ko,은,142,topic particle (after consonant)
8,ko,▁,129,word boundary
9,ko,가,100,subject particle (after vowel)


## Week 36.3
Verify programmatically that every answerable item’s answer equals the substring beginning at answer start, and report the number checked and any failures.

In [28]:
# train
ans = df_train[df_train["answerable"]]
ok = [context[start:start + len(answer)] == answer
      for context, start, answer in zip(ans["context"], ans["answer_start"], ans["answer"])]
print(f"train: answer: {len(ok)} - fail: {ok.count(False)}")

train: answer: 5972 - fail: 0


In [29]:
# val
ans = df_val[df_val["answerable"]]
ok = [context[start:start + len(answer)] == answer
      for context, start, answer in zip(ans["context"], ans["answer_start"], ans["answer"])]
print(f"validation: answer: {len(ok)} - fail: {ok.count(False)}")

validation: answer: 991 - fail: 0


## Week 36.4
Implement and evaluate two answerability baselines: (i) the majority-class baseline estimated from the training split and (ii) a deterministic rule-based classifier that uses only the question and context. The rule may use tokenisation, lexical features or machine translation, but no labelled validation examples or trained answerability/QA model. Discuss what information the rule can and cannot exploit in this cross-lingual setting.

In [30]:
def evaluate_answerability(df, pred_column):
    rows = []
    for lang in ["all"] + LANGUAGES:
        part = df if lang == "all" else df[df["lang"] == lang]
        report = classification_report(part["answerable"], part[pred_column], labels=[True, False],
                                       target_names=["answerable", "unanswerable"], zero_division=0, output_dict=True)
        rows.append({
            "lang": lang,
            "n": len(part),
            "n_unans": int(report["unanswerable"]["support"]),
            "ans_p": report["answerable"]["precision"],
            "ans_r": report["answerable"]["recall"],
            "ans_f1": report["answerable"]["f1-score"],
            "unans_p": report["unanswerable"]["precision"],
            "unans_r": report["unanswerable"]["recall"],
            "unans_f1": report["unanswerable"]["f1-score"],
            "macro_f1": report["macro avg"]["f1-score"],
            "accuracy": report["accuracy"],
        })
    return pd.DataFrame(rows).round(3)

### (i) Majority-class baseline

In [31]:
majority_class = df_train["answerable"].mode()[0]
print("majority class:", majority_class)

majority class: True


In [32]:
# train
df_train["pred_majority"] = majority_class
evaluate_answerability(df_train, "pred_majority")

,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,6335,363,0.943,1.0,0.971,0.0,0.0,0.0,0.485,0.943
1,ar,2558,255,0.900,1.0,0.948,0.0,0.0,0.0,0.474,0.900
2,ko,2422,63,0.974,1.0,0.987,0.0,0.0,0.0,0.493,0.974
3,te,1355,45,0.967,1.0,0.983,0.0,0.0,0.0,0.492,0.967


In [33]:
# val
df_val["pred_majority"] = majority_class
evaluate_answerability(df_val, "pred_majority")

,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,1155,164,0.858,1.0,0.924,0.0,0.0,0.0,0.462,0.858
1,ar,415,52,0.875,1.0,0.933,0.0,0.0,0.0,0.467,0.875
2,ko,356,19,0.947,1.0,0.973,0.0,0.0,0.0,0.486,0.947
3,te,384,93,0.758,1.0,0.862,0.0,0.0,0.0,0.431,0.758


### (ii) Rule-based classifier
Answerable if the question and the context share at least one XLM-R token (question marks and the bare word boundary `▁` excluded), otherwise unanswerable.

In [34]:
# claude
PUNCTUATION_TOKENS = {"▁", "?", "▁?", "؟", "▁؟"}

def rule_answerable(question, context):
    question_tokens = set(xlm_tokeniser.tokenize(question)) - PUNCTUATION_TOKENS
    context_tokens = set(xlm_tokeniser.tokenize(context))
    return len(question_tokens & context_tokens) > 0

In [35]:
# train
df_train["pred_rule"] = [rule_answerable(question, context) for question, context in zip(df_train["question"], df_train["context"])]
evaluate_answerability(df_train, "pred_rule")

,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,6335,363,0.962,0.051,0.098,0.058,0.967,0.110,0.104,0.104
1,ar,2558,255,0.990,0.045,0.086,0.104,0.996,0.188,0.137,0.140
2,ko,2422,63,0.953,0.060,0.113,0.025,0.889,0.048,0.081,0.082
3,te,1355,45,0.938,0.047,0.089,0.032,0.911,0.061,0.075,0.075


In [36]:
# val
df_val["pred_rule"] = [rule_answerable(question, context) for question, context in zip(df_val["question"], df_val["context"])]
evaluate_answerability(df_val, "pred_rule")

,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,1155,164,0.792,0.038,0.073,0.139,0.939,0.242,0.158,0.166
1,ar,415,52,1.000,0.022,0.043,0.128,1.000,0.227,0.135,0.145
2,ko,356,19,0.842,0.047,0.090,0.047,0.842,0.090,0.090,0.090
3,te,384,93,0.667,0.048,0.090,0.237,0.925,0.377,0.233,0.260


In [39]:
baselines = []
for model, column in [("majority", "pred_majority"), ("rule", "pred_rule")]:
    for split, df in [("train", df_train), ("val", df_val)]:
        baselines.append(evaluate_answerability(df, column).assign(model=model, split=split))
pd.concat(baselines).to_json(ROOT_DIR / "w36_baselines.json", orient="records", indent=2)

In [40]:
w36_baselines = pd.read_json(ROOT_DIR / "w36_baselines.json")
w36_baselines[w36_baselines["split"] == "val"]

,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy,model,split
4,all,1155,164,0.858,1.000,0.924,0.000,0.000,0.000,0.462,0.858,majority,val
5,ar,415,52,0.875,1.000,0.933,0.000,0.000,0.000,0.467,0.875,majority,val
6,ko,356,19,0.947,1.000,0.973,0.000,0.000,0.000,0.486,0.947,majority,val
7,te,384,93,0.758,1.000,0.862,0.000,0.000,0.000,0.431,0.758,majority,val
12,all,1155,164,0.792,0.038,0.073,0.139,0.939,0.242,0.158,0.166,rule,val
13,ar,415,52,1.000,0.022,0.043,0.128,1.000,0.227,0.135,0.145,rule,val
14,ko,356,19,0.842,0.047,0.090,0.047,0.842,0.090,0.090,0.090,rule,val
15,te,384,93,0.667,0.048,0.090,0.237,0.925,0.377,0.233,0.260,rule,val
